# Lab 52: The Invisible Sixth Sense

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 6, *The Embodied Avatar*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-52.ipynb)

**What you will do:** measure how far your fingertip-touching accuracy degrades after spinning, compare it with the book's typical ranges, and build a simple robot-arm inverse-kinematics solver to see why it never drifts the way your own proprioception does.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

The one-foot, eyes-closed balance test (Step 4) and the after-spinning version of the reaching
test (the "Variation" paragraph) carry a real risk of losing your balance and falling. Do both
near a wall, a heavy piece of furniture, or a friend you can hold on to, on a soft or carpeted
floor if possible, and stop immediately if you feel dizzy or unsteady. Skip the spinning-chair
variation if you have any inner-ear or balance condition, or simply feel unwell doing it.

## The experiment in the book

In Lab 52 you closed your eyes and touched your nose, matched the position of one arm with the
other, and brought your two index fingers together in front of you -- all without seeing your
hands. The book explains that this relies on proprioception: signals from receptors in your
muscles, tendons and joints that tell the brain where your body parts are, integrated with
vestibular information to keep the body schema accurate. Spinning in a chair briefly disrupts
this calibration, making the finger-touching task noticeably less accurate afterwards.

## Record your results

Do the finger-touching test from the "Variation" paragraph several times before spinning, and
several times just after spinning three times in a chair (with the safety precautions above).
Each time, measure the gap between your fingertips in centimetres when you open your eyes -- a
ruler or tape measure held nearby helps.

In [ ]:
finger_gap_cm = pd.DataFrame({
    "condition": ["before spinning"] * 4 + ["after spinning"] * 4,
    "trial": [1, 2, 3, 4] * 2,
    "gap (cm)": [1.5, 1.0, 2.0, 1.5, 6.0, 7.5, 5.0, 8.0],   # example data: replace with your own measurements
})
display(finger_gap_cm)
summary = finger_gap_cm.groupby("condition")["gap (cm)"].agg(["mean", "std"])
display(summary)
summary["mean"].plot.barh(xerr=summary["std"], color=["steelblue", "firebrick"], figsize=(6, 2.4))
plt.xlabel("finger-touching gap (cm)"); plt.show()

## Compare

The book gives two approximate benchmarks: without disturbance, most people bring their
fingertips together within about 1-2 cm; after spinning, misses of 5-10 cm are typical. Check
your own numbers against both ranges.

In [ ]:
before_mean = summary.loc["before spinning", "mean"]
after_mean = summary.loc["after spinning", "mean"]
print(f"Before spinning: {before_mean:.1f} cm (book's typical range: 1-2 cm)")
print(f"After spinning:  {after_mean:.1f} cm (book's typical range: 5-10 cm)")

if 1 <= before_mean <= 2:
    print("Your undisturbed accuracy is squarely in the book's typical range.")
elif before_mean < 1:
    print("You were even more accurate than the book's typical range -- a good sign of precise")
    print("proprioception, though a handful of trials is not a lot of data.")
else:
    print("You were less accurate than the book's typical range before spinning; try a few more trials to")
    print("see if this was a fluke, or whether you were rushing the movement.")

if 5 <= after_mean <= 10:
    print("Your post-spinning accuracy matches the book's typical range for vestibular disruption.")
else:
    print("Your post-spinning miss was outside the book's typical range -- spins vary in how disorienting")
    print("they are, and three quick spins may have disrupted you more or less than the book's readers.")

## The AI side

The book compares proprioception to inverse kinematics (IK) in robotics: working out what
joint angles will place a hand at a target point. Below, a simple two-link arm (like a
shoulder and elbow) computes the joint angles needed to reach a target using closed-form
geometry, then checks the reach by simulating the resulting hand position -- the robot's
equivalent of your finger-touching test, but with no proprioceptive noise at all.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L1, L2 = 30.0, 25.0   # upper-arm and forearm lengths, cm

def solve_ik(x, y):
    d2 = x ** 2 + y ** 2
    cos_elbow = np.clip((d2 - L1 ** 2 - L2 ** 2) / (2 * L1 * L2), -1, 1)
    elbow = np.arccos(cos_elbow)
    shoulder = np.arctan2(y, x) - np.arctan2(L2 * np.sin(elbow), L1 + L2 * np.cos(elbow))
    return shoulder, elbow

def forward_kinematics(shoulder, elbow):
    x1, y1 = L1 * np.cos(shoulder), L1 * np.sin(shoulder)
    x2 = x1 + L2 * np.cos(shoulder + elbow)
    y2 = y1 + L2 * np.sin(shoulder + elbow)
    return (0, x1, x2), (0, y1, y2)

targets = [(40, 10), (20, 30), (10, -20)]
fig, ax = plt.subplots(figsize=(5, 5))
for tx, ty in targets:
    shoulder, elbow = solve_ik(tx, ty)
    xs, ys = forward_kinematics(shoulder, elbow)
    ax.plot(xs, ys, "o-", label=f"target ({tx},{ty})")
    ax.plot(tx, ty, "kx")
ax.set_aspect("equal"); ax.legend(fontsize=8); ax.set_title("Two-link arm reaching by inverse kinematics")
plt.show()

print("The arm's computed hand position lands exactly on each target -- IK has an exact geometric solution")
print("for a rigid two-link arm, with no noise and no need for sensory feedback once the joint lengths are")
print("known.")

Unlike your own proprioceptive reaching, which drifted by several centimetres after spinning,
this IK solver reaches every target exactly, every time, because it never has to sense
anything -- it is simply told the lengths of its "limbs" and the target. Real robot arms lose
this precision the moment their joint encoders have noise, their links flex slightly, or an
unmodelled load changes the arm's true geometry, which is when they need something closer to
real proprioceptive feedback rather than one-shot geometry.

## Questions to think about

1. Your after-spinning error was larger than your before-spinning error (if it was). The IK solver above never makes this kind of error. What information does your brain have to compensate for that a two-link IK solver of known, fixed lengths does not?
2. The book distinguishes proprioception from inverse kinematics by saying that proprioception includes real-time feedback about forces and velocities, not just joint angles. Which part of your finger-touching task most depends on that kind of dynamic feedback, rather than just knowing where your joints are?
3. If you changed L1 or L2 in the code (to model, say, a child's shorter arm) without updating solve_ik's use of them, what would happen to the computed reach? What is the biological equivalent of this kind of mismatch?
4. Ian Waterman, described in the book, lost his proprioception entirely and had to watch his limbs to move them. If a robot arm's joint-angle sensors failed completely, could it still reach a visible target using only a camera? What would be different about how it did so, compared with your own visually-guided reaching?

## Challenge

Try the nose-touching test (Step 1) with your dominant hand and your non-dominant hand
separately, several trials each, and measure or rate how close your fingertip lands to your
nose. See whether proprioceptive accuracy differs between your two hands.

In [ ]:
# Example data: replace with your own trials, in cm from fingertip to nose tip, eyes closed.
hand = ["dominant"] * 4 + ["non-dominant"] * 4
miss_cm = [0.5, 0.0, 1.0, 0.5, 1.5, 2.0, 1.0, 2.5]   # example data

results = pd.DataFrame({"hand": hand, "miss (cm)": miss_cm})
display(results)
summary = results.groupby("hand")["miss (cm)"].agg(["mean", "std"])
display(summary)
print("A small difference between hands is common and does not necessarily mean much on its own; a large,")
print("consistent difference would be a more interesting proprioceptive asymmetry to investigate further.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-52.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")